# Bangladesh and the Flying Geese hypothesis — reproducible panel analysis

**Data source:** the repository's existing `wdi_data.csv` file. This notebook does not call the World Bank API. Put the CSV in the notebook's working directory (or set `DATA_PATH` below).

The goal is to assess whether Bangladesh's industrial structure, trade and investment trends are consistent with the Flying Geese (FG) hypothesis. Aggregate indicators can test broad consistency, but cannot prove product-level transfer of comparative advantage.

**Data handling:** no interpolation, no missing-to-zero conversion, country-year duplicates are checked, and each result reports sample size. WDI CSV exports are commonly wide (one column per year); the loader below supports standard WDI wide format and already-long CSV formats.

**References**
- Akamatsu, K. (1962). A historical pattern of economic growth in developing countries. *The Developing Economies*, 1(S1), 3–25.
- Dowling, M. & Cheang, C.T. (2000). Shifting comparative advantage in Asia: new tests of the flying geese model. *Journal of Asian Economics*, 11(4), 443–463.
- World Bank (2018), [The Future of Manufacturing-Led Development in East Asia](https://documents1.worldbank.org/curated/en/116191523612461208/pdf/125189-PUB-PUBLIC-docdate-4-11-18.pdf).
- [WDI data and documentation](https://datatopics.worldbank.org/world-development-indicators/).

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

DATA_PATH = Path("wdi_data.csv")  # repository filename is lowercase
if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH.resolve()}. Place wdi_data.csv beside this notebook. "
        f"CSV files currently visible: {[p.name for p in candidates]}"
    )

raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded:", DATA_PATH, "| shape:", raw.shape)
display(raw.head(3))
print("Columns:", raw.columns.tolist()[:20])

## 1. Standardize the repository CSV

The loader identifies standard WDI exports (`Country Name`, `Country Code`, `Indicator Name`, `Indicator Code`, year columns) and long tables. It keeps the chosen Flying Geese countries and the indicators needed for industrial structure, trade and investment. If the CSV uses different labels, the diagnostic below shows its columns rather than silently substituting data.

In [ ]:
COUNTRIES = {
    "BGD":"Bangladesh", "CHN":"China", "IDN":"Indonesia", "JPN":"Japan",
    "KOR":"Korea, Rep.", "MYS":"Malaysia", "PHL":"Philippines",
    "SGP":"Singapore", "VNM":"Viet Nam"
}
INDICATORS = {
    "NV.IND.MANF.ZS":"manufacturing_gdp",
    "NV.IND.TOTL.ZS":"industry_gdp",
    "NV.SRV.TOTL.ZS":"services_gdp",
    "NV.AGR.TOTL.ZS":"agriculture_gdp",
    "NE.EXP.GNFS.ZS":"exports_gdp",
    "NE.IMP.GNFS.ZS":"imports_gdp",
    "NE.TRD.GNFS.ZS":"trade_gdp",
    "BX.KLT.DINV.WD.GD.ZS":"fdi_gdp",
    "NE.GDI.FTOT.ZS":"gfcf_gdp",
    "NE.GDI.TOTL.ZS":"gross_capital_gdp",
    "NY.GDP.MKTP.KD.ZG":"gdp_growth",
    "NV.IND.MANF.KD.ZG":"manufacturing_growth",
    "TX.VAL.TECH.MF.ZS":"hightech_exports_share",
}
# Normalize headers but preserve original names for identifying year columns.
lower = {str(c).strip().lower(): c for c in raw.columns}
def col(*names):
    for name in names:
        if name.lower() in lower: return lower[name.lower()]
    return None

cc = col("Country Code", "iso3", "iso3c", "countrycode")
cn = col("Country Name", "country", "country_name")
ic = col("Indicator Code", "indicator_code", "series code", "series_code")
inn = col("Indicator Name", "indicator", "indicator_name", "series name")
year_cols = [c for c in raw.columns if str(c).strip().isdigit() and 1900 <= int(str(c).strip()) <= 2100]

if cc and ic and year_cols:  # standard WDI wide file
    sub = raw[raw[cc].astype(str).isin(COUNTRIES) & raw[ic].astype(str).isin(INDICATORS)].copy()
    if sub.empty:
        raise ValueError("WDI wide format detected, but no target country/indicator rows matched. Check Country Code and Indicator Code columns.")
    long = sub.melt(id_vars=[cc] + ([cn] if cn else []) + [ic],
                    value_vars=year_cols, var_name="year", value_name="value")
    long = long.rename(columns={cc:"iso3", ic:"indicator_code"})
    long["country"] = long["iso3"].map(COUNTRIES)
    long["variable"] = long["indicator_code"].map(INDICATORS)
    long["year"] = pd.to_numeric(long["year"], errors="coerce")
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
elif cc and (col("year") is not None) and (ic or inn):
    # Long-format CSV
    yc = col("year"); vc = col("value", "obs_value", "observation value")
    if vc is None: raise ValueError("Long-format file detected but no value column found.")
    long = raw.copy().rename(columns={cc:"iso3", yc:"year", vc:"value"})
    if ic:
        long = long[long[ic].astype(str).isin(INDICATORS)].copy()
        long["variable"] = long[ic].map(INDICATORS)
    else:
        long["variable"] = long[inn].map({v:k for k,v in INDICATORS.items()})
    long = long[long["iso3"].astype(str).isin(COUNTRIES)].copy()
    long["country"] = long["iso3"].map(COUNTRIES)
    long["year"] = pd.to_numeric(long["year"], errors="coerce")
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
else:
    raise ValueError("CSV layout not recognized. Expected WDI wide columns (Country Code, Indicator Code, year columns) or a long table (country code, year, indicator, value). Inspect raw.columns above.")

long = long[long["variable"].notna() & long["year"].notna()].copy()
dup = long.duplicated(["iso3","year","variable"], keep=False)
print("Target observations:", len(long), "| duplicated country-year-indicator rows:", int(dup.sum()))
if dup.any():
    display(long.loc[dup].sort_values(["iso3","year","variable"]).head(20))
    raise ValueError("Duplicate country-year-indicator rows found; resolve before analysis rather than silently aggregating.")
panel = long.pivot(index=["iso3","country","year"], columns="variable", values="value").reset_index()
panel = panel.sort_values(["iso3","year"]).reset_index(drop=True)
panel["ntc"] = (panel.get("exports_gdp") - panel.get("imports_gdp")) / (panel.get("exports_gdp") + panel.get("imports_gdp"))
panel["exports_imports_ratio"] = panel.get("exports_gdp") / panel.get("imports_gdp").where(panel.get("imports_gdp").ne(0))
print("Panel rows:", len(panel), "| countries:", panel.country.nunique(), "| years:", panel.year.min(), "to", panel.year.max())
display(pd.DataFrame({"nonmissing":panel.notna().sum(), "coverage_pct":(100*panel.notna().mean()).round(1)}))

## 2. Visual EDA

GDP shares describe composition, not absolute scale. High-tech exports (% manufactured exports) is especially sparse in many WDI vintages; its sample size must be shown. The country-year lines below use available observations only.

In [ ]:
plot_vars = [
    ("manufacturing_gdp","Manufacturing value added (% GDP)"),
    ("industry_gdp","Industry value added (% GDP)"),
    ("exports_gdp","Exports (% GDP)"),
    ("imports_gdp","Imports (% GDP)"),
    ("gfcf_gdp","Gross fixed capital formation (% GDP)"),
    ("fdi_gdp","FDI net inflows (% GDP)"),
    ("hightech_exports_share","High-tech exports (% manufactured exports)"),
    ("manufacturing_growth","Manufacturing value-added growth (%)")
]
for v, title in plot_vars:
    if v not in panel or panel[v].notna().sum() == 0:
        print("Skipping unavailable indicator:", v); continue
    fig, ax = plt.subplots(figsize=(12,5))
    sns.lineplot(data=panel, x="year", y=v, hue="country", marker="o", ax=ax)
    ax.set_title(title); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

display(panel.groupby("country")[[v for v,_ in plot_vars if v in panel]].agg(["count","min","max"]).round(2))

## 3. Common-period comparison and trend tests

Country comparisons use overlapping years for each indicator. Trend significance is corrected across country-indicator combinations with Benjamini–Hochberg false-discovery-rate control.

In [ ]:
comparison = []
for v,_ in plot_vars:
    if v not in panel: continue
    bd = panel.loc[panel.country.eq("Bangladesh"),["year",v]].dropna().rename(columns={v:"Bangladesh"})
    for peer in sorted(set(panel.country)-{"Bangladesh"}):
        pp = panel.loc[panel.country.eq(peer),["year",v]].dropna().rename(columns={v:peer})
        z = bd.merge(pp,on="year").sort_values("year")
        if len(z):
            comparison.append({"indicator":v,"peer":peer,"n_common":len(z),"first_year":int(z.year.min()),
                               "last_year":int(z.year.max()),"BD_change":z.Bangladesh.iloc[-1]-z.Bangladesh.iloc[0],
                               "peer_change":z[peer].iloc[-1]-z[peer].iloc[0]})
common = pd.DataFrame(comparison)
display(common.round(3))

trend_rows=[]
for country in sorted(panel.country.dropna().unique()):
    for v,_ in plot_vars:
        if v not in panel: continue
        z=panel.loc[panel.country.eq(country),["year",v]].dropna()
        if len(z)>=10:
            rho,p=stats.spearmanr(z.year,z[v])
            trend_rows.append({"country":country,"indicator":v,"n":len(z),"rho_year":rho,
                               "linear_slope_per_year":np.polyfit(z.year,z[v],1)[0],"p_raw":p})
trends=pd.DataFrame(trend_rows)
if not trends.empty:
    trends["p_BH"]=multipletests(trends.p_raw,method="fdr_bh")[1]
display(trends.sort_values("p_BH").round(4))

## 4. Candidate structural breaks

This is an exploratory segmented-trend scan, not a substitute for a pre-specified Chow test or a validated multiple-break procedure. Each candidate requires at least 10 observations on either side; the best raw p-value is reported with Bonferroni correction across candidate dates. A date is not called a confirmed turning point unless the adjusted result and alternative specifications agree.

In [ ]:
def scan_break(z, variable, min_side=10):
    z=z[["year",variable]].dropna().sort_values("year").copy()
    if len(z)<2*min_side+3: return pd.DataFrame()
    out=[]
    candidate_years=sorted(z.year.unique())[min_side-1:-min_side]
    for year in candidate_years:
        q=z.copy(); q["t"]=q.year-q.year.min(); q["post"]=(q.year>year).astype(int); q["tpost"]=q.t*q.post
        fit=smf.ols(f'Q("{variable}") ~ t + post + tpost',data=q).fit(cov_type="HC3")
        try:
            w=fit.wald_test("post=0, tpost=0",scalar=True)
            out.append({"break_year":int(year),"p_raw":float(w.pvalue),"n_before":int((q.year<=year).sum()),"n_after":int((q.year>year).sum())})
        except Exception: pass
    res=pd.DataFrame(out)
    if len(res): res["p_Bonferroni"]=multipletests(res.p_raw,method="bonferroni")[1]
    return res
break_rows=[]
for country in sorted(panel.country.dropna().unique()):
    for v in ["manufacturing_gdp","exports_gdp","imports_gdp","gfcf_gdp","fdi_gdp"]:
        if v not in panel: continue
        r=scan_break(panel.loc[panel.country.eq(country)],v)
        if len(r):
            best=r.loc[r.p_raw.idxmin()].to_dict()
            best.update(country=country,indicator=v,n_series=int(panel.loc[panel.country.eq(country),v].notna().sum()))
            break_rows.append(best)
breaks=pd.DataFrame(break_rows)
display(breaks.sort_values("p_Bonferroni").round(4) if len(breaks) else "No series have enough observations for the minimum regime sizes.")

## 5. Two-way fixed effects: associations, not causality

Country and year fixed effects control for stable country differences and common year shocks. One-year lagged predictors reduce immediate simultaneity but do not solve endogeneity. There are only nine countries, so clustered standard errors are approximate; interpret p-values cautiously.

In [ ]:
for v in ["fdi_gdp","gfcf_gdp","exports_gdp","imports_gdp","hightech_exports_share","manufacturing_gdp"]:
    if v in panel: panel[v+"_lag1"]=panel.groupby("iso3")[v].shift(1)
specs=[
    ("manufacturing_gdp",["fdi_gdp_lag1","gfcf_gdp_lag1","exports_gdp_lag1"]),
    ("exports_gdp",["manufacturing_gdp_lag1","fdi_gdp_lag1","gfcf_gdp_lag1"]),
    ("gfcf_gdp",["fdi_gdp_lag1","manufacturing_gdp_lag1"])
]
reg_rows=[]
for y,xs in specs:
    if y not in panel or any(x not in panel for x in xs): continue
    z=panel[[y]+xs+["iso3","year"]].dropna()
    if len(z)<50 or z.iso3.nunique()<5: continue
    fit=smf.ols(f'{y} ~ '+' + '.join(xs)+' + C(iso3) + C(year)',data=z).fit(
        cov_type="cluster",cov_kwds={"groups":z.iso3})
    for x in xs:
        reg_rows.append({"outcome":y,"predictor":x,"coefficient":fit.params[x],"cluster_se":fit.bse[x],
                         "p_raw":fit.pvalues[x],"n":int(fit.nobs),"countries":z.iso3.nunique()})
reg=pd.DataFrame(reg_rows)
if len(reg): reg["p_BH"]=multipletests(reg.p_raw,method="fdr_bh")[1]
display(reg.round(4))

## 6. ADF/KPSS stationarity diagnostics

ADF's null is a unit root; KPSS's null is stationarity. Disagreement is informative. These are individual-series diagnostics, not panel unit-root tests, and structural breaks can affect both. Avoid running standard level regressions on trending variables without checking integration and cointegration assumptions.

In [ ]:
st=[]
for country in sorted(panel.country.dropna().unique()):
    for v in ["manufacturing_gdp","exports_gdp","imports_gdp","fdi_gdp","gfcf_gdp","manufacturing_growth"]:
        if v not in panel: continue
        s=panel.loc[panel.country.eq(country),["year",v]].dropna().sort_values("year")[v]
        if len(s)<15: continue
        for form,series in [("level",s),("first_difference",s.diff().dropna())]:
            if len(series)<12: continue
            try: adf_p=adfuller(series,autolag="AIC",regression="ct" if form=="level" else "c")[1]
            except Exception: adf_p=np.nan
            try: kpss_p=kpss(series,regression="ct" if form=="level" else "c",nlags="auto")[1]
            except Exception: kpss_p=np.nan
            st.append({"country":country,"indicator":v,"form":form,"n":len(series),"ADF_p":adf_p,"KPSS_p":kpss_p})
stationarity=pd.DataFrame(st)
display(stationarity.round(4))

## 7. Bangladesh versus earlier geese: lead-lag similarity

Lagged rank correlations are exploratory. They do not prove that one economy passed a product line to another; shared global shocks, trends and serial correlation can create similar patterns. We require at least 12 common observations and correct p-values across the tested comparisons.

In [ ]:
ll=[]
for v in ["manufacturing_gdp","exports_gdp","hightech_exports_share","gfcf_gdp"]:
    if v not in panel: continue
    for peer in ["China","Korea, Rep.","Viet Nam"]:
        if peer not in set(panel.country): continue
        for lag in [0,5,10]:
            for form in ["levels","annual_change"]:
                a=panel.loc[panel.country.eq("Bangladesh")].set_index("year")[v]
                b=panel.loc[panel.country.eq(peer)].set_index("year")[v]
                if form=="annual_change": a=a.diff(); b=b.diff()
                z=pd.concat([a.rename("bd"),b.rename("peer")],axis=1)
                z["peer_lag"]=z.peer.shift(lag); z=z[["bd","peer_lag"]].dropna()
                if len(z)>=12:
                    rho,p=stats.spearmanr(z.bd,z.peer_lag)
                    ll.append({"indicator":v,"peer":peer,"peer_lag_years":lag,"form":form,"n":len(z),"rho":rho,"p_raw":p})
leadlag=pd.DataFrame(ll)
if len(leadlag): leadlag["p_BH"]=multipletests(leadlag.p_raw,method="fdr_bh")[1]
display(leadlag.sort_values("p_BH").round(4) if len(leadlag) else "No lead-lag comparison has 12 or more observations.")

## 8. Evidence-based conclusion and limitations

Use the results above to distinguish three claims:

1. **Industrial deepening:** manufacturing output or share changes in Bangladesh.
2. **Trade upgrading:** Bangladesh gains comparative advantage in more complex or different products, not merely higher trade/GDP.
3. **Flying Geese sequence:** product-level evidence shows the changing production/export composition and relative advantage consistent with sequential upgrading across economies.

This WDI notebook can assess (1) and parts of (2), but cannot by itself establish (3). If the evidence is mixed, the correct conclusion is “partially consistent with, but insufficient to establish, the Flying Geese hypothesis.”

**Recommended supplementary datasets:** [UN Comtrade](https://comtradeplus.un.org/) or [CEPII BACI](https://www.cepii.fr/CEPII/en/bdd_modele/presentation.asp?id=37) for HS product-level trade and revealed comparative advantage; [UNIDO](https://stat.unido.org/) for manufacturing subsectors and employment. WDI CSVs can be downloaded from the World Bank's [WDI portal](https://datatopics.worldbank.org/world-development-indicators/).